# T4 (BM25) — tách từ và dựng chỉ mục BM25 trên Kaggle CPU (không tốn hạn mức GPU)

**Cài đặt notebook:** Accelerator = **None** (CPU), Internet = **On** (cài pyvi, jieba).
**Add Input:** `mir-code`, `mir-data` (cần `chunks/structure/`).
**Cách chạy:** *Save Version* → *Save & Run All (Commit)*. Ước tính khoảng 1 giờ với 4 tiến trình, từ tốc độ pyvi
khoảng 340k và jieba khoảng 360k ký tự/s/lõi đo trên máy cá nhân (chưa đo trên Kaggle). Làm song song được với T3.
**Đầu ra:** `index/structure/bm25/` → tab *Output* của version → *New Dataset* tên **`mir-bm25`**.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây; phần còn lại giống nhau ở mọi notebook.
NB_VERSION = 't4-v2 (05/10/2026)'
OVERRIDES = {}

import glob, os, shutil, sys
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')          # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mir/__init__.py', recursive=True), key=len)
assert hits, 'Chưa thêm dataset mir-code (Add Input → Your Datasets)'
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)                     # luôn chép lại bản mir-code mới nhất
shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mir.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, OVERRIDES, corpus='full', pip='pyvi jieba')

In [ ]:
# (2) Tách từ từng part: pyvi cho tiếng Việt, jieba cho tiếng Trung (part đã xong được bỏ qua)
sh(f'python -m mir.index bm25-tokenize --config {K.cfg}', 'bm25_tokenize')

In [ ]:
# (3) Thống kê toàn kho + ma trận BM25 theo shard; xoá dữ liệu tạm tf/ cho output gọn
sh(f'python -m mir.index bm25-build --config {K.cfg}', 'bm25_build')
print(open(f'{K.work}/index/structure/bm25/stats.json').read())
shutil.rmtree(f'{K.work}/index/structure/bm25/tf', ignore_errors=True)
report('t4_bm25')